# 11 - Export Pairs Data
Kumukuha ng H1 historical data mula sa MT5 para sa lahat ng 7 major pairs,
nag-co-compute ng ML features, at nag-sa-save ng CSV files sa `data/` folder.

**Output files:**
- `data/EURUSD_H1_ML.csv`
- `data/GBPUSD_H1_ML.csv`
- `data/USDJPY_H1_ML.csv`
- `data/USDCHF_H1_ML.csv`
- `data/AUDUSD_H1_ML.csv`
- `data/USDCAD_H1_ML.csv`
- `data/NZDUSD_H1_ML.csv`

Pagkatapos nito, i-run mo ang:
```
python src/ml/train.py
```
para ma-train ang XGBoost model para sa bawat pair.

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

# Add project root to path
ROOT = Path().absolute()
if ROOT.name == 'notebooks' or not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)

print(f'Project root : {ROOT}')
print(f'Data folder  : {DATA_DIR}')


## Config — Edit here if needed

In [ ]:
# ── Settings ─────────────────────────────────────────────────────────
SYMBOLS   = ['EURUSD', 'GBPUSD', 'USDJPY', 'USDCHF', 'AUDUSD', 'USDCAD', 'NZDUSD']
TIMEFRAME = 'H1'
BARS      = 5000   # ~5000 H1 bars ≈ 7 months of data
OVERWRITE = False   # Set False to skip already-exported pairs

print(f'Symbols   : {SYMBOLS}')
print(f'Timeframe : {TIMEFRAME}')
print(f'Bars      : {BARS}')


## Step 1 — Connect to MT5

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(ROOT / '.env')

from src.data.provider import DataProvider

provider = DataProvider(
    mt5_login    = int(os.getenv('MT5_LOGIN', '0')) or None,
    mt5_password = os.getenv('MT5_PASSWORD', ''),
    mt5_server   = os.getenv('MT5_SERVER', ''),
)

connected = provider.connect_mt5()
print(f'MT5 connected: {connected}')

if not connected:
    print()
    print('WARNING: MT5 not connected — will use SIMULATED data instead.')
    print('Simulated data is okay for testing the pipeline,')
    print('but train your final models with REAL MT5 data!')


## Step 2 — Feature Engineering Function
Same features as `src/ml/train.py` — Target column = next-bar direction.

In [ ]:
def _session(h: int) -> int:
    if 7  <= h < 12: return 0   # London
    if 12 <= h < 16: return 3   # Overlap
    if 16 <= h < 20: return 1   # NY
    return 2                     # Asian


def add_features_and_target(df: pd.DataFrame) -> pd.DataFrame:
    """Compute all ML features + binary Target column."""
    df = df.copy()
    c  = df['close']

    # ── Trend ─────────────────────────────────────────────────────────
    sma20 = c.rolling(20).mean()
    sma50 = c.rolling(50).mean()
    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    macd  = ema12 - ema26
    msig  = macd.ewm(span=9, adjust=False).mean()

    # ── RSI ───────────────────────────────────────────────────────────
    delta = c.diff()
    gain  = delta.clip(lower=0).ewm(span=14, adjust=False).mean()
    loss  = (-delta.clip(upper=0)).ewm(span=14, adjust=False).mean()
    rsi   = 100 - (100 / (1 + gain / loss.replace(0, np.nan)))

    # ── Bollinger ─────────────────────────────────────────────────────
    bb_sma = c.rolling(20).mean()
    bb_std = c.rolling(20).std()
    bb_top = bb_sma + 2 * bb_std
    bb_bot = bb_sma - 2 * bb_std

    # ── ATR ───────────────────────────────────────────────────────────
    hl  = df['high'] - df['low']
    hc  = (df['high'] - c.shift()).abs()
    lc  = (df['low']  - c.shift()).abs()
    tr  = pd.concat([hl, hc, lc], axis=1).max(axis=1)
    atr = tr.rolling(14).mean()

    vol_col = 'tick_volume' if 'tick_volume' in df.columns else 'volume'
    vol_ma  = df[vol_col].rolling(20).mean()

    # ── Assign features ───────────────────────────────────────────────
    df['pct_from_sma20']    = (c - sma20) / sma20 * 100
    df['pct_from_sma50']    = (c - sma50) / sma50 * 100
    df['rsi']               = rsi
    df['rsi_lag1']          = rsi.shift(1)
    df['macd_hist']         = macd - msig
    df['bb_position']       = (c - bb_bot) / (bb_top - bb_bot).replace(0, np.nan)
    df['bb_width']          = (bb_top - bb_bot) / bb_sma * 100
    df['atr_pct']           = atr / c * 100
    df['return_1']          = c.pct_change(1)  * 100
    df['return_5']          = c.pct_change(5)  * 100
    df['body_ratio']        = (df['close'] - df['open']).abs() / hl.replace(0, np.nan)
    df['volume_ratio']      = df[vol_col] / vol_ma.replace(0, np.nan)

    # ── Time features ─────────────────────────────────────────────────
    df['session'] = df.index.hour.map(_session)
    df['dow']     = df.index.dayofweek

    # ── Composite SMC features ────────────────────────────────────────
    df['momentum_alignment'] = (
        (df['macd_hist']   > 0).astype(int) +
        (df['rsi']         > 50).astype(int) +
        (df['bb_position'] > 0.5).astype(int)
    )
    df['trend_strength'] = df[['pct_from_sma20', 'pct_from_sma50']].abs().mean(axis=1)
    df['vol_spike']      = (df['volume_ratio'] > 1.5).astype(int)
    df['rsi_extreme']    = ((df['rsi'] < 30) | (df['rsi'] > 70)).astype(int)
    df['atr_regime']     = (
    df['atr_regime']     = (
        pd.qcut(df['atr_pct'].fillna(df['atr_pct'].median()), 3, labels=[0, 1, 2], duplicates='drop')
        .astype(float).fillna(1).astype(int)
    )
    )

    # ── Target: 1 = next bar closes higher (Win), 0 = lower (Loss) ───
    df['Target'] = (c.shift(-1) > c).astype(int)

    return df


print('Feature engineering function ready.')


## Step 3 — Fetch, Compute Features, Save CSV per Pair

In [ ]:
results = []

for symbol in SYMBOLS:
    out_path = DATA_DIR / f'{symbol}_H1_ML.csv'

    if out_path.exists() and not OVERWRITE:
        print(f'[SKIP]  {symbol} — file already exists ({out_path.name})')
        results.append({'symbol': symbol, 'status': 'skipped', 'rows': None})
        continue

    print(f'[{symbol}] Fetching {BARS} {TIMEFRAME} bars...')
    price_data = provider.fetch_rates(symbol, TIMEFRAME, BARS)

    if price_data is None or price_data.data.empty:
        print(f'[{symbol}] ERROR: no data returned — skipping')
        results.append({'symbol': symbol, 'status': 'error', 'rows': 0})
        continue

    df = price_data.data.copy()

    # Ensure DatetimeIndex for time features
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)

    # Compute features + target
    df = add_features_and_target(df)

    # Drop last row (Target = NaN because no next bar)
    df = df.iloc[:-1]

    # Save
    df.to_csv(out_path, index_label='time')
    rows = len(df)
    print(f'[{symbol}] Saved {rows:,} rows → {out_path.name}')
    results.append({'symbol': symbol, 'status': 'ok', 'rows': rows})

    time.sleep(0.3)  # Small delay between MT5 requests

print()
print('Done!')


## Summary

In [ ]:
print(f'{'Symbol':<10} {'Status':<10} {'Rows':>8}')
print('-' * 32)
for r in results:
    rows_str = f"{r['rows']:,}" if r['rows'] else '—'
    print(f"{r['symbol']:<10} {r['status']:<10} {rows_str:>8}")

# Check which files are ready for training
print()
print('Files ready for training:')
ready = []
for sym in SYMBOLS:
    p = DATA_DIR / f'{sym}_H1_ML.csv'
    if p.exists():
        rows = len(pd.read_csv(p))
        print(f'  ✓ {p.name}  ({rows:,} rows)')
        ready.append(sym)
    else:
        print(f'  ✗ {sym}_H1_ML.csv  (missing)')

print()
print('Next step — train ML models:')
print(f'  python src/ml/train.py')
print(f'  (will train {len(ready)} models: {", ".join(ready)})')
